### Sanity check bc_mpralm results with feedback from Pia
- Investigate Control example which we expect to be significant:
    - Liang https://docs.google.com/document/d/1_ngAV5ccj07aI4aKfMGYPUtdmVJ1PGvV2dxgZPsqXlU/edit?tab=t.0#heading=h.tlollb7eimle
- How many variants go into bc_MPRAlm
  - Without any variant controls: 42129
  - With the variant controls without any matching problem: 41474
- Have they all at least 10 barcodes?
- The following things were checked in another script (analyze_NGN2)
- How do the MPRAlm results look like? (compare log2FC and p-values)
- How many dna and rna counts do we have from these results?



In [2]:
import pandas as pd
import math
import yaml

# config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "../../global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf


#### Control example:
- read the MPRAsnakeflow results
- compute the activity of the sequences on your own 
    - check if you have enough barcodes 
    - check if the barcodes have enough measurements 
    - sum over all barcodes

In [3]:
mprasnakeflow_assigned_barocdes_counts_path = config['files']['creating']['mprasnakeflow_resequencing_bbmap10_assigned_barcodes_unique_variants']
mprasnakeflow_assigned_barocdes_counts_path = config['files']['creating']['mprasnakeflow_resequencing_bbmap35_assigned_barcodes']
mprasnakeflow_assigned_barocdes_counts_df = pd.read_csv(mprasnakeflow_assigned_barocdes_counts_path, sep="\t")
mprasnakeflow_assigned_barocdes_counts_df
#

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,GGCCTCTTCGGTCAG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,6.0,18.0,1.0,12.0,7.0,11.0
1,AATACCCAAGAACCC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,1.0,2.0,1.0,4.0,NaN,NaN
2,TGGGTGCGCCAATTA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,1.0,1.0,NaN,NaN
3,ATATCAAGACGCGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,12.0,21.0,8.0,14.0,5.0,23.0
4,TAAATATCATAAGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,7.0,9.0,2.0,11.0,4.0,13.0
...,...,...,...,...,...,...,...,...
6223264,TTTCGACCGGATGCG,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3.0,11.0,3.0,10.0,3.0,9.0
6223265,TGGCGAGCACTGGCC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,5.0,12.0,1.0,12.0
6223266,TGAGGCTTGTTGGGC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3.0,9.0,3.0,4.0,2.0,7.0
6223267,TATTGTCAAGCGGGA,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,1.0,16.0,2.0,12.0,3.0,7.0


In [4]:
# get liang
mprasnakeflow_assigned_barocdes_counts_df['tmp_label'] = mprasnakeflow_assigned_barocdes_counts_df['name'].apply(hf.get_label)

In [ ]:
variant_map = pd.read_csv(config['files']['final_design']['variant_table'], sep="\t")
variant_map
variant_map['tmp_label'] = variant_map['ID'].apply(hf.get_label)


Number of variants: ref: 8, alt: 8


GC_Kircher: 

In [43]:
variant_map_kircher = variant_map.loc[variant_map['tmp_label'] == 'GC_Kircher'].copy()
variant_map_kircher
nref = variant_map_kircher['REF'].nunique()
nalt = variant_map_kircher['ALT'].nunique()
print(f'Number of variants: ref: {nref}, alt: {nalt}')
kircher_dna_rna_count_df = mprasnakeflow_assigned_barocdes_counts_df.loc[mprasnakeflow_assigned_barocdes_counts_df['tmp_label'] == 'GC_Kircher'].copy()

Number of variants: ref: 5, alt: 198


In [44]:
kircher_dna_rna_count_df.groupby('name').size()

name
GC_Kircher:ALT_NC000001.11|109274794|C|T|KircherControls~NC000001.11|109274836|C|T|KircherControls~NC000001.11|109274840|A|C|KircherControls~NC000001.11|109274845|C|A|KircherControls~NC000001.11|109274846|T|G|KircherControls~NC000001.11|109274852|T|G|KircherControls~NC000001.11|109274857|T|C|KircherControls~NC000001.11|109274860|C|G|KircherControls~NC000001.11|109274865|C|A|KircherControls~NC000001.11|109274869|G|C|KircherControls~NC000001.11|109274884|G|C|KircherControls~NC000001.11|109274885|T|C|KircherControls~NC000001.11|109274886|C|A|KircherControls~NC000001.11|109274887|A|G|KircherControls~NC000001.11|109274888|T|G|KircherControls~NC000001.11|109274892|T|A|KircherControls~NC000001.11|109274908|T|G|KircherControls~NC000001.11|109274910|C|A|KircherControls~NC000001.11|109274912|G|T|KircherControls~NC000001.11|109274917|G|T|KircherControls~NC000001.11|109274922|T|C|KircherControls~NC000001.11|109274923|G|C|KircherControls~NC000001.11|109274924|G|C|KircherControls~NC000001.11|10

In [45]:
# compute summed dna count and rna count
dna_columns = [col for col in kircher_dna_rna_count_df.columns if col.startswith("dna_count")]
rna_columns = [col for col in kircher_dna_rna_count_df.columns if col.startswith("rna_count")]

kircher_dna_rna_count_df["dna_sum"] = kircher_dna_rna_count_df[dna_columns].sum(axis=1, skipna=True)
kircher_dna_rna_count_df["rna_sum"] = kircher_dna_rna_count_df[rna_columns].sum(axis=1, skipna=True)
kircher_dna_rna_count_df

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,tmp_label,dna_sum,rna_sum
192923,GTTGAGACCAACTGG,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,1.0,12.0,1.0,8.0,5.0,13.0,GC_Kircher,7.0,33.0
192924,ATAGACATGGCGGGT,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,4.0,16.0,6.0,15.0,5.0,16.0,GC_Kircher,15.0,47.0
192925,TGCGGCTGCCGTTAC,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,4.0,7.0,1.0,7.0,5.0,4.0,GC_Kircher,10.0,18.0
192926,ACTTAACGTAGAATG,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,1.0,4.0,NaN,NaN,1.0,1.0,GC_Kircher,2.0,5.0
192927,ACGGGGATTCCTAGG,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,2.0,6.0,4.0,2.0,4.0,5.0,GC_Kircher,10.0,13.0
...,...,...,...,...,...,...,...,...,...,...,...
203420,AAAGGGGTCAAACGA,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,NaN,NaN,3.0,14.0,NaN,NaN,GC_Kircher,3.0,14.0
203421,GTCCTGTCTGCGAAA,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,NaN,NaN,8.0,28.0,NaN,NaN,GC_Kircher,8.0,28.0
203422,GACATGAATCGAACG,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,NaN,NaN,6.0,15.0,NaN,NaN,GC_Kircher,6.0,15.0
203423,CGGTGGAGACTACTT,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,NaN,NaN,6.0,22.0,NaN,NaN,GC_Kircher,6.0,22.0


In [46]:
# aggregate counts over the barcodes
kircher_element_activity = kircher_dna_rna_count_df.groupby("name")[["dna_sum", "rna_sum"]].sum().reset_index()
# log rna and dna
kircher_element_activity["rna_log2"] = kircher_element_activity["rna_sum"].apply(math.log2)
kircher_element_activity["dna_log2"] = kircher_element_activity["dna_sum"].apply(math.log2)

# Compute activity (rna_dna_ratio)
kircher_element_activity["rna_dna_ratio"] = kircher_element_activity["rna_sum"] / kircher_element_activity["dna_sum"]
kircher_element_activity["rna_dna_ratio_log2"] = kircher_element_activity["rna_log2"] - kircher_element_activity["dna_log2"]

# Display the result
kircher_element_activity

,name,dna_sum,rna_sum,rna_log2,dna_log2,rna_dna_ratio,rna_dna_ratio_log2
0,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,194.0,552.0,9.108524,7.599913,2.845361,1.508612
1,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,161.0,426.0,8.734710,7.330917,2.645963,1.403793
2,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,309.0,920.0,9.845490,8.271463,2.977346,1.574027
3,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,84.0,282.0,8.139551,6.392317,3.357143,1.747234
4,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,224.0,612.0,9.257388,7.807355,2.732143,1.450033
...,...,...,...,...,...,...,...
173,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,287.0,742.0,9.535275,8.164907,2.585366,1.370368
174,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,358.0,1006.0,9.974415,8.483816,2.810056,1.490599
175,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,638.0,1803.0,10.816184,9.317413,2.826019,1.498771
176,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,141.0,435.0,8.764872,7.139551,3.085106,1.625320


In [48]:
variant_map_kircher_ratios = variant_map_kircher.merge(kircher_element_activity[['name', 'rna_dna_ratio', 'rna_dna_ratio_log2']], left_on='REF', right_on='name', how='inner')
# rename the column to reference
variant_map_kircher_ratios.rename(columns={'rna_dna_ratio': 'rna_dna_ratio_ref', 'rna_dna_ratio_log2': 'rna_dna_ratio_log2_ref'}, inplace=True)
variant_map_kircher_ratios.drop(columns=['name'], inplace=True)
variant_map_kircher_ratios = variant_map_kircher_ratios.merge(kircher_element_activity[['name', 'rna_dna_ratio', 'rna_dna_ratio_log2']], left_on='ALT', right_on='name', how='inner')
variant_map_kircher_ratios.rename(columns={'rna_dna_ratio': 'rna_dna_ratio_alt', 'rna_dna_ratio_log2': 'rna_dna_ratio_log2_alt'}, inplace=True)
variant_map_kircher_ratios.drop(columns=['name'], inplace=True)

variant_map_kircher_ratios # 173 variants

,ID,Region,REF,ALT,tmp_label,rna_dna_ratio_ref,rna_dna_ratio_log2_ref,rna_dna_ratio_alt,rna_dna_ratio_log2_alt
0,GC_Kircher:NC000001_11_109274794_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.585366,1.370368,2.845361,1.508612
1,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.585366,1.370368,2.645963,1.403793
2,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.810056,1.490599,2.736721,1.452448
3,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.585366,1.370368,2.977346,1.574027
4,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.810056,1.490599,2.636923,1.398855
...,...,...,...,...,...,...,...,...,...
168,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.608696,1.383329,2.641834,1.401540
169,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,3.085106,1.625320,2.888889,1.530515
170,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.608696,1.383329,2.723183,1.445294
171,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.608696,1.383329,1.156863,0.210218


In [52]:
# variant effects:
variant_map_kircher_ratios['log2_alt_ref'] = variant_map_kircher_ratios['rna_dna_ratio_alt'] / variant_map_kircher_ratios['rna_dna_ratio_ref']
variant_map_kircher_ratios['log2_alt_ref'] = variant_map_kircher_ratios['log2_alt_ref'].apply(lambda x: math.log2(x))
variant_map_kircher_ratios['log2_alt_ref_log2_sub'] = variant_map_kircher_ratios['rna_dna_ratio_log2_alt'] - variant_map_kircher_ratios['rna_dna_ratio_log2_ref']
print(f"min: {round(variant_map_kircher_ratios['log2_alt_ref_log2_sub'].min(),3)} - max: {round(variant_map_kircher_ratios['log2_alt_ref_log2_sub'].max(),3)}")
variant_map_kircher_ratios.sort_values(by='log2_alt_ref_log2_sub')

min: -1.173 - max: 0.377


,ID,Region,REF,ALT,tmp_label,rna_dna_ratio_ref,rna_dna_ratio_log2_ref,rna_dna_ratio_alt,rna_dna_ratio_log2_alt,log2_alt_ref,log2_alt_ref_log2_sub
171,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.608696,1.383329,1.156863,0.210218,-1.173111,-1.173111
20,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.585366,1.370368,2.092308,1.065095,-0.305273,-0.305273
162,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.826019,1.498771,2.315615,1.211395,-0.287376,-0.287376
103,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.810056,1.490599,2.353535,1.234830,-0.255769,-0.255769
164,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,3.085106,1.625320,2.625767,1.392739,-0.232581,-0.232581
...,...,...,...,...,...,...,...,...,...,...,...
3,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.585366,1.370368,2.977346,1.574027,0.203659,0.203659
127,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,3.085106,1.625320,3.566667,1.834576,0.209256,0.209256
123,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.826019,1.498771,3.314943,1.728984,0.230213,0.230213
56,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,2.810056,1.490599,3.385417,1.759333,0.268735,0.268735


GC_Liang

In [ ]:
variant_map_liang = variant_map.loc[variant_map['tmp_label'] == 'GC_Liang'].copy()
variant_map_liang
nref = variant_map_liang['REF'].nunique()
nalt = variant_map_liang['ALT'].nunique()
print(f'Number of variants: ref: {nref}, alt: {nalt}')
liang_dna_rna_count_df = mprasnakeflow_assigned_barocdes_counts_df.loc[mprasnakeflow_assigned_barocdes_counts_df['tmp_label'] == 'GC_Liang'].copy()

In [20]:
# get number of ref and alt (only 7 complete variants)
liang_oligos = set(liang_dna_rna_count_df['name'].to_list())
nliang_ref = len([x for x in liang_oligos if 'REF_' in x])
nliang_alt = len([x for x in liang_oligos if 'ALT_' in x])
print(f'Number of variants: ref: {nliang_ref}, alt: {nliang_alt}')

Number of variants: ref: 8, alt: 7


In [21]:
variant_map_liang.head()

,ID,Region,REF,ALT,tmp_label
46397,GC_Liang:rs2125358,GC_Liang:rs2125358|Liang_fwd_tile1-1,GC_Liang:REF_rs2125358|Liang_fwd_tile1-1,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,GC_Liang
46398,GC_Liang:rs17882077,GC_Liang:rs17882077|Liang_fwd_tile1-1,GC_Liang:REF_rs17882077|Liang_fwd_tile1-1,GC_Liang:ALT_rs17882077|Liang_fwd_tile1-1_rs17...,GC_Liang
46399,GC_Liang:rs10502466,GC_Liang:rs10502466|Liang_fwd_tile1-1,GC_Liang:REF_rs10502466|Liang_fwd_tile1-1,GC_Liang:ALT_rs10502466|Liang_fwd_tile1-1_rs10...,GC_Liang
46400,GC_Liang:rs1036014,GC_Liang:rs1036014|Liang_fwd_tile1-1,GC_Liang:REF_rs1036014|Liang_fwd_tile1-1,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,GC_Liang
46401,GC_Liang:rs2838227,GC_Liang:rs2838227|Liang_fwd_tile1-1,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,GC_Liang:ALT_rs2838227|Liang_fwd_tile1-1_rs283...,GC_Liang


In [22]:
liang_dna_rna_count_df.groupby('name').size()

name
GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs1036014      140
GC_Liang:ALT_rs10502466|Liang_fwd_tile1-1_rs10502466     64
GC_Liang:ALT_rs10939614|Liang_fwd_tile1-1_rs10939614     72
GC_Liang:ALT_rs17603855|Liang_fwd_tile1-1_rs17603855    107
GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs2125358       36
GC_Liang:ALT_rs2530731|Liang_fwd_tile1-1_rs2530731       53
GC_Liang:ALT_rs2838227|Liang_fwd_tile1-1_rs2838227       17
GC_Liang:REF_rs1036014|Liang_fwd_tile1-1                 95
GC_Liang:REF_rs10502466|Liang_fwd_tile1-1               104
GC_Liang:REF_rs10939614|Liang_fwd_tile1-1               113
GC_Liang:REF_rs17603855|Liang_fwd_tile1-1               121
GC_Liang:REF_rs17882077|Liang_fwd_tile1-1                11
GC_Liang:REF_rs2125358|Liang_fwd_tile1-1                 14
GC_Liang:REF_rs2530731|Liang_fwd_tile1-1                 28
GC_Liang:REF_rs2838227|Liang_fwd_tile1-1                 19
dtype: int64

In [ ]:
# GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs2125358 + GC_Liang:REF_rs2125358|Liang_fwd_tile1-1


In [23]:
example_alt = mprasnakeflow_assigned_barocdes_counts_df.loc[mprasnakeflow_assigned_barocdes_counts_df['name'] == 'GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs2125358']
example_ref = mprasnakeflow_assigned_barocdes_counts_df.loc[mprasnakeflow_assigned_barocdes_counts_df['name'] == 'GC_Liang:REF_rs2125358|Liang_fwd_tile1-1']

In [24]:
example_alt.shape[0] # 36
example_alt
# example_ref.shape[0] # 14
# example_ref

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,tmp_label
203808,GGCTTATCCCTCACA,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,7.0,24.0,5.0,47.0,5.0,43.0,GC_Liang
203809,TAAACACCCCCACGC,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,NaN,NaN,2.0,5.0,2.0,3.0,GC_Liang
203810,TCGGATACTAACCTC,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,NaN,NaN,3.0,5.0,4.0,4.0,GC_Liang
203811,CGACATGCCTATATG,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,3.0,21.0,3.0,25.0,7.0,19.0,GC_Liang
203812,TGTGAGCGTGCGGTT,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,3.0,17.0,1.0,9.0,3.0,9.0,GC_Liang
203813,TACCTGTTGAGCCCT,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,12.0,35.0,6.0,34.0,13.0,35.0,GC_Liang
203814,AGCGTATGAACCGTA,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,2.0,13.0,NaN,NaN,5.0,11.0,GC_Liang
203815,CAAAACTCGATGCAC,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,NaN,NaN,NaN,NaN,6.0,3.0,GC_Liang
203816,ATAGGGCTCGCCCAG,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,3.0,30.0,8.0,18.0,7.0,23.0,GC_Liang
203817,GCAGAATTGTCGCTA,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,12.0,44.0,9.0,45.0,13.0,47.0,GC_Liang


In [25]:
liang_dna_rna_count_df.columns

Index(['Barcode', 'name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3', 'tmp_label'],
      dtype='object')

In [26]:
# compute summed dna count and rna count
dna_columns = [col for col in liang_dna_rna_count_df.columns if col.startswith("dna_count")]
rna_columns = [col for col in liang_dna_rna_count_df.columns if col.startswith("rna_count")]

liang_dna_rna_count_df["dna_sum"] = liang_dna_rna_count_df[dna_columns].sum(axis=1, skipna=True)
liang_dna_rna_count_df["rna_sum"] = liang_dna_rna_count_df[rna_columns].sum(axis=1, skipna=True)
liang_dna_rna_count_df

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,tmp_label,dna_sum,rna_sum
203425,GCGTCTCACTGTACG,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,1.0,5.0,1.0,7.0,1.0,2.0,GC_Liang,3.0,14.0
203426,TTAACCCGCTATTCG,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,2.0,9.0,3.0,10.0,4.0,10.0,GC_Liang,9.0,29.0
203427,CCGCTATTGCCAATG,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,11.0,42.0,11.0,40.0,14.0,40.0,GC_Liang,36.0,122.0
203428,GTACCTACAGATAAC,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,NaN,NaN,1.0,13.0,3.0,9.0,GC_Liang,4.0,22.0
203429,GAGGACTTTCTATCC,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,3.0,21.0,7.0,16.0,4.0,19.0,GC_Liang,14.0,56.0
...,...,...,...,...,...,...,...,...,...,...,...
204414,CGTAAGCTACCTCCT,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,4.0,15.0,4.0,10.0,1.0,7.0,GC_Liang,9.0,32.0
204415,GACGATGAATTATAT,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,6.0,7.0,NaN,NaN,NaN,NaN,GC_Liang,6.0,7.0
204416,AAGGAGGTCGCTGAA,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,3.0,16.0,5.0,10.0,4.0,14.0,GC_Liang,12.0,40.0
204417,GGCGACTGTACGTAT,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,4.0,18.0,5.0,11.0,4.0,7.0,GC_Liang,13.0,36.0


In [33]:
# aggregate counts over the barcodes
liang_element_activity = liang_dna_rna_count_df.groupby("name")[["dna_sum", "rna_sum"]].sum().reset_index()
# log rna and dna
liang_element_activity["rna_log2"] = liang_element_activity["rna_sum"].apply(math.log2)
liang_element_activity["dna_log2"] = liang_element_activity["dna_sum"].apply(math.log2)

# Compute activity (rna_dna_ratio)
liang_element_activity["rna_dna_ratio"] = liang_element_activity["rna_sum"] / liang_element_activity["dna_sum"]
liang_element_activity["rna_dna_ratio_log2"] = liang_element_activity["rna_log2"] - liang_element_activity["dna_log2"]

# Display the result
liang_element_activity

,name,dna_sum,rna_sum,rna_log2,dna_log2,rna_dna_ratio,rna_dna_ratio_log2
0,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,1510.0,4639.0,12.179598,10.560333,3.072185,1.619265
1,GC_Liang:ALT_rs10502466|Liang_fwd_tile1-1_rs10...,500.0,971.0,9.923327,8.965784,1.942000,0.957543
2,GC_Liang:ALT_rs10939614|Liang_fwd_tile1-1_rs10...,542.0,1282.0,10.324181,9.082149,2.365314,1.242032
3,GC_Liang:ALT_rs17603855|Liang_fwd_tile1-1_rs17...,614.0,966.0,9.915879,9.262095,1.573290,0.653785
4,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,430.0,1490.0,10.541097,8.748193,3.465116,1.792904
5,GC_Liang:ALT_rs2530731|Liang_fwd_tile1-1_rs253...,612.0,1429.0,10.480790,9.257388,2.334967,1.223402
6,GC_Liang:ALT_rs2838227|Liang_fwd_tile1-1_rs283...,79.0,146.0,7.189825,6.303781,1.848101,0.886044
7,GC_Liang:REF_rs1036014|Liang_fwd_tile1-1,1202.0,3753.0,11.873829,10.231221,3.122296,1.642607
8,GC_Liang:REF_rs10502466|Liang_fwd_tile1-1,718.0,1419.0,10.470659,9.487840,1.976323,0.982819
9,GC_Liang:REF_rs10939614|Liang_fwd_tile1-1,939.0,2265.0,11.145295,9.874981,2.412141,1.270314


In [34]:
variant_map_liang_ratios = variant_map_liang.merge(liang_element_activity[['name', 'rna_dna_ratio', 'rna_dna_ratio_log2']], left_on='REF', right_on='name')
# rename the column to reference
variant_map_liang_ratios.rename(columns={'rna_dna_ratio': 'rna_dna_ratio_ref', 'rna_dna_ratio_log2': 'rna_dna_ratio_log2_ref'}, inplace=True)
variant_map_liang_ratios.drop(columns=['name'], inplace=True)
variant_map_liang_ratios = variant_map_liang_ratios.merge(liang_element_activity[['name', 'rna_dna_ratio', 'rna_dna_ratio_log2']], left_on='ALT', right_on='name')
variant_map_liang_ratios.rename(columns={'rna_dna_ratio': 'rna_dna_ratio_alt', 'rna_dna_ratio_log2': 'rna_dna_ratio_log2_alt'}, inplace=True)
variant_map_liang_ratios.drop(columns=['name'], inplace=True)

variant_map_liang_ratios

,ID,Region,REF,ALT,tmp_label,rna_dna_ratio_ref,rna_dna_ratio_log2_ref,rna_dna_ratio_alt,rna_dna_ratio_log2_alt
0,GC_Liang:rs2125358,GC_Liang:rs2125358|Liang_fwd_tile1-1,GC_Liang:REF_rs2125358|Liang_fwd_tile1-1,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,GC_Liang,3.123223,1.643035,3.465116,1.792904
1,GC_Liang:rs10502466,GC_Liang:rs10502466|Liang_fwd_tile1-1,GC_Liang:REF_rs10502466|Liang_fwd_tile1-1,GC_Liang:ALT_rs10502466|Liang_fwd_tile1-1_rs10...,GC_Liang,1.976323,0.982819,1.942000,0.957543
2,GC_Liang:rs1036014,GC_Liang:rs1036014|Liang_fwd_tile1-1,GC_Liang:REF_rs1036014|Liang_fwd_tile1-1,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,GC_Liang,3.122296,1.642607,3.072185,1.619265
3,GC_Liang:rs2838227,GC_Liang:rs2838227|Liang_fwd_tile1-1,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,GC_Liang:ALT_rs2838227|Liang_fwd_tile1-1_rs283...,GC_Liang,2.269461,1.182350,1.848101,0.886044
4,GC_Liang:rs10939614,GC_Liang:rs10939614|Liang_fwd_tile1-1,GC_Liang:REF_rs10939614|Liang_fwd_tile1-1,GC_Liang:ALT_rs10939614|Liang_fwd_tile1-1_rs10...,GC_Liang,2.412141,1.270314,2.365314,1.242032
5,GC_Liang:rs17603855,GC_Liang:rs17603855|Liang_fwd_tile1-1,GC_Liang:REF_rs17603855|Liang_fwd_tile1-1,GC_Liang:ALT_rs17603855|Liang_fwd_tile1-1_rs17...,GC_Liang,1.780255,0.832084,1.573290,0.653785
6,GC_Liang:rs2530731,GC_Liang:rs2530731|Liang_fwd_tile1-1,GC_Liang:REF_rs2530731|Liang_fwd_tile1-1,GC_Liang:ALT_rs2530731|Liang_fwd_tile1-1_rs253...,GC_Liang,2.671096,1.417432,2.334967,1.223402


In [54]:
variant_map_liang_ratios['log2_alt_ref'] = variant_map_liang_ratios['rna_dna_ratio_alt'] / variant_map_liang_ratios['rna_dna_ratio_ref']
variant_map_liang_ratios['log2_alt_ref'] = variant_map_liang_ratios['log2_alt_ref'].apply(lambda x: math.log2(x))
variant_map_liang_ratios['log2_alt_ref_log2_sub'] = variant_map_liang_ratios['rna_dna_ratio_log2_alt'] - variant_map_liang_ratios['rna_dna_ratio_log2_ref']
print(f"min: {round(variant_map_liang_ratios['log2_alt_ref_log2_sub'].min(),3)} - max: {round(variant_map_liang_ratios['log2_alt_ref_log2_sub'].max(),3)}")
variant_map_liang_ratios.sort_values(by='log2_alt_ref_log2_sub')

min: -0.296 - max: 0.15


,ID,Region,REF,ALT,tmp_label,rna_dna_ratio_ref,rna_dna_ratio_log2_ref,rna_dna_ratio_alt,rna_dna_ratio_log2_alt,log2_alt_ref,log2_alt_ref_log2,log2_alt_ref_log2_sub
3,GC_Liang:rs2838227,GC_Liang:rs2838227|Liang_fwd_tile1-1,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,GC_Liang:ALT_rs2838227|Liang_fwd_tile1-1_rs283...,GC_Liang,2.269461,1.182350,1.848101,0.886044,-0.296306,-0.296306,-0.296306
6,GC_Liang:rs2530731,GC_Liang:rs2530731|Liang_fwd_tile1-1,GC_Liang:REF_rs2530731|Liang_fwd_tile1-1,GC_Liang:ALT_rs2530731|Liang_fwd_tile1-1_rs253...,GC_Liang,2.671096,1.417432,2.334967,1.223402,-0.194030,-0.194030,-0.194030
5,GC_Liang:rs17603855,GC_Liang:rs17603855|Liang_fwd_tile1-1,GC_Liang:REF_rs17603855|Liang_fwd_tile1-1,GC_Liang:ALT_rs17603855|Liang_fwd_tile1-1_rs17...,GC_Liang,1.780255,0.832084,1.573290,0.653785,-0.178299,-0.178299,-0.178299
4,GC_Liang:rs10939614,GC_Liang:rs10939614|Liang_fwd_tile1-1,GC_Liang:REF_rs10939614|Liang_fwd_tile1-1,GC_Liang:ALT_rs10939614|Liang_fwd_tile1-1_rs10...,GC_Liang,2.412141,1.270314,2.365314,1.242032,-0.028282,-0.028282,-0.028282
1,GC_Liang:rs10502466,GC_Liang:rs10502466|Liang_fwd_tile1-1,GC_Liang:REF_rs10502466|Liang_fwd_tile1-1,GC_Liang:ALT_rs10502466|Liang_fwd_tile1-1_rs10...,GC_Liang,1.976323,0.982819,1.942000,0.957543,-0.025276,-0.025276,-0.025276
2,GC_Liang:rs1036014,GC_Liang:rs1036014|Liang_fwd_tile1-1,GC_Liang:REF_rs1036014|Liang_fwd_tile1-1,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,GC_Liang,3.122296,1.642607,3.072185,1.619265,-0.023342,-0.023342,-0.023342
0,GC_Liang:rs2125358,GC_Liang:rs2125358|Liang_fwd_tile1-1,GC_Liang:REF_rs2125358|Liang_fwd_tile1-1,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,GC_Liang,3.123223,1.643035,3.465116,1.792904,0.149868,0.149868,0.149868


In [30]:
print('No significant variant effect found according to the counts')
# use outlier detection
import pandas as pd
import numpy as np

# Example function to detect outliers using z-score method
def is_outlier(series):
    z_scores = (series - series.mean(skipna=True)) / series.std(skipna=True)
    return abs(z_scores) > 3


# Identify RNA columns
rna_columns = [col for col in liang_dna_rna_count_df.columns if "rna" in col]

# Detect outliers in any RNA column
outlier_mask = liang_dna_rna_count_df[rna_columns].apply(is_outlier, axis=0).any(axis=1)

# Filter out rows where any RNA count is an outlier
filtered_var_df = liang_dna_rna_count_df[~outlier_mask]
print(liang_dna_rna_count_df.shape[0])
filtered_var_df.shape[0]

No significant variant effect found according to the counts
994


967

variant_map_liang_ratios['log2_alt_ref'] = variant_map_liang_ratios

#### BCalm Numbers: 

In [42]:
config["files"]["creating"]["toptable_bcMPRAlm"]

'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/projects/bc_tradeoff/results/bc_MPRAlm/test_controls_concat/toptable_standard_with_variant_controls_no_downsamplingNGN2.feather'

In [43]:
# bcMPRAlm_table = pd.read_csv(config["files"]["toptable_standard_control_table"], sep="\t")
bcMPRAlm_table = pd.read_csv(config["files"]["creating"]["toptable_bcMPRAlm"], sep="\t")


print("%s analyzable variants"%(bcMPRAlm_table.shape[0])) # 41284 rows; low config: 42137 rows; 34683 with 10 barcodes

34683 analyzable variants


In [ ]:
### how many variants in bc mpralm

### do all of them have 10 barcodes?
# load the sequence to barcode table
sequence_barcode_association = pd.read_csv(config["files"]["final_design"]["sequence_to_barcode_assignment"], sep="\t", header=None)
sequence_barcode_association.columns = ["barcode", "ID", "alignment_info", "association_info"]
sequence_barcode_association


,barcode,ID,alignment_info,association_info
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,15;270M;NM:i:0;MD:Z:270;60,5/5
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,6/7
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,15;270M;NM:i:0;MD:Z:270;6,9/10
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,15;270M;NM:i:0;MD:Z:270;6,7/7
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,7/8
...,...,...,...,...
6305461,TTTTTTTTTTGACCG,cardiac_neuro_cava_random:ALT_CPS1|ENSG0000002...,15;270M;NM:i:0;MD:Z:270;6,32/33
6305462,TTTTTTTTTTGACGA,cardiac_neuro_cava_random:ALT_ACTN2|ENSG000000...,15;270M;NM:i:0;MD:Z:270;6,11/11
6305463,TTTTTTTTTTGCACA,cardiac_neuro_cava_random:ALT_CARD11|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,22/26
6305464,TTTTTTTTTTGCTAA,cardiac_neuro_cava_random:REF_NR4A2|ENSG000001...,15;270M;NM:i:0;MD:Z:270;6,6/6


In [50]:
mpralm_with_barcode = bcMPRAlm_table.merge(sequence_barcode_association, left_on="variant_id", right_on="ID", how="left")
# does not work because matching without variant region map not possible

In [ ]:
# how many unique IDs
mpralm_with_barcode["ID"].nunique()

75004

In [46]:
sequence_barcode_association.columns = ["barcode", "ID", "alignment_info", "association_info"]
sequence_barcode_association

,barcode,ID,alignment_info,association_info
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,15;270M;NM:i:0;MD:Z:270;60,5/5
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,6/7
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,15;270M;NM:i:0;MD:Z:270;6,9/10
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,15;270M;NM:i:0;MD:Z:270;6,7/7
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,7/8
...,...,...,...,...
6305461,TTTTTTTTTTGACCG,cardiac_neuro_cava_random:ALT_CPS1|ENSG0000002...,15;270M;NM:i:0;MD:Z:270;6,32/33
6305462,TTTTTTTTTTGACGA,cardiac_neuro_cava_random:ALT_ACTN2|ENSG000000...,15;270M;NM:i:0;MD:Z:270;6,11/11
6305463,TTTTTTTTTTGCACA,cardiac_neuro_cava_random:ALT_CARD11|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,22/26
6305464,TTTTTTTTTTGCTAA,cardiac_neuro_cava_random:REF_NR4A2|ENSG000001...,15;270M;NM:i:0;MD:Z:270;6,6/6


In [47]:
# count for each ID how many barcodes are associated
barcode_count = sequence_barcode_association.groupby("ID").count()
barcode_count = barcode_count.reset_index()
barcode_count = barcode_count[['ID', 'barcode']]
barcode_count

,ID,barcode
0,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,150
1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,132
2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,88
3,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,122
4,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,72
...,...,...
74999,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,48
75000,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,109
75001,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,36
75002,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,57


In [48]:

# check if all ids have 10 barcodes
bc_number_threshold = 10
barcode_count['label'] = barcode_count['ID'].apply(lambda x: x.split(":")[0])
sequences_below_10_barcodes = barcode_count[barcode_count['barcode'] < bc_number_threshold]
print(barcode_count[barcode_count['barcode'] >= bc_number_threshold].shape[0])
print(f"IDs with less than {bc_number_threshold} barcodes:", len(sequences_below_10_barcodes))
# print(sequences_below_10_barcodes) # 4380 rows with less than 10 barcodes; 731 with less than 2 barcodes

# # check their distribution
print(sequences_below_10_barcodes['label'].value_counts())

# cardiac_below_10_bcs = sequences_below_10_barcodes[sequences_below_10_barcodes['label'] == "cardiac_neuro_cava_random"]
# # check how many variants and how many regions
# cardiac_below_10_bcs['sequence_type'] = cardiac_below_10_bcs['ID'].apply(lambda x: "variant" if ":REF_" in x or ":ALT_" in x else "region")
# cardiac_below_10_bcs['sequence_type'].value_counts()
# cardiac_below_10_bcs['chrom_pos_ref_alt'] = cardiac_below_10_bcs['ID'].apply(lambda x: x.split("|")[-1])
# cardiac_below_10_bcs


70624
IDs with less than 10 barcodes: 4380
label
cardiac_neuro_cava_random      3790
MK                              157
C_positive_heart_AB             145
C_negative_neuron_NP             49
C_negative_heart_MK              32
C_positive_neuron_CD             31
C_positive_neuron_NP             26
GC_Selvarajan                    25
GC_Kircher                       23
C_negative_neuron_MK             22
C_positive_heart_MK              22
C_SLEA                           19
GC_Vista                         13
GC_Mendelian_variants             8
C_positive_heart_CAD              4
GC_GABA_Chengyu                   3
GC_DNase_positive                 3
GC_Liang                          2
C_positive_neuron_MK              2
GC_DNase_positive_shuffeled       2
GC_Glut_Chengyu                   1
GC_Mohlke                         1
Name: count, dtype: int64


In [ ]:
### compare mpralm log2fc and p-values